# Phi-4 Arrhenius Re-Analysis: Wilson CIs, Delta-Approx BON-16, Power Analysis

This notebook is a demo of the **zero-API-cost re-analysis** of the Phi-4 Arrhenius Kinetics of LLM Inference experiment.

## What this artifact does

Using 30 valid OC (only-correctable) instances from TIGER-Lab/MMLU-Pro, it computes:

1. **Delta-Approx BON-16 Accuracy**: Uses the single-call logit-gap approximation (`T_thresh = Delta/ln(16) + 0.3`, snapped to temperature grid) to estimate Best-of-16 accuracy. This quantifies the efficiency-accuracy tradeoff vs. full Arrhenius regression.
2. **Wilson 95% Confidence Intervals**: Computes Wilson score CIs for the Table 1 fractions (N=4,8,16,32), replacing the point estimates with properly bounded intervals.
3. **Scatter Plot**: T_operating (Arrhenius) vs T_pref (empirical argmax), showing Spearman ρ=0.674.
4. **Power Analysis**: Formal Fisher z-transform power analysis for the observed ρ=0.674 at n=30.
5. **Deployment Algorithm**: Three-case routing pseudocode (greedy/OC/robust-error).
6. **Dataset Discrepancy Note**: Clarifies the catalysis set overlap with the main 500-item set.

**Demo note**: The demo uses a 5-instance subset of the 30 OC valid instances for metrics 1 & 3. Metrics 2, 4, 5, 6 operate on pre-computed aggregate statistics and are unaffected by the subset size.

In [ ]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# loguru is NOT pre-installed on Colab
_pip('loguru==0.7.3')

# Core scientific packages — pre-installed on Colab, install locally only
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'scipy==1.16.3', 'matplotlib==3.10.0')

In [ ]:
import json
import math
import sys
from pathlib import Path

from loguru import logger
import numpy as np
from scipy.stats import norm
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

logger.remove()
logger.add(sys.stdout, level="INFO", format="{time:HH:mm:ss}|{level:<7}|{message}")

## Data Loading

The demo loads from GitHub (when running on Colab after deployment) with a local file fallback for running locally.

In [ ]:
GITHUB_DATA_URL = "https://raw.githubusercontent.com/AMGrobelnik/ai-invention-ce2af9-arrhenius-kinetics-of-llm-inference-acti/main/round-3/evaluation-1/demo/mini_demo_data.json"
import json, os

def load_data():
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception: pass
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f: return json.load(f)
    raise FileNotFoundError("Could not load mini_demo_data.json")

In [ ]:
data = load_data()
print(f"Loaded data with {len(data['datasets'][0]['examples'])} examples")
print(f"Metadata keys: {list(data['metadata'].keys())}")

## Config

Tunable parameters for the re-analysis. The demo uses the 5 OC examples in the mini dataset. The original experiment used `N_VALID_OC = 30`.

In [ ]:
# Temperature grid used for snapping T_operating
GRID = [0.05, 0.1, 0.2, 0.3, 0.5, 0.7, 1.0]

# Best-of-N parameter
N_BON = 16

# Expected number of valid OC instances in the FULL dataset (for assertions)
# Original: 30 — demo uses 5 (mini subset)
N_VALID_OC_EXPECTED = 5  # full original: 30

## Core Helper Functions

These functions implement the statistical tools used across all metrics:
- `wilson_ci`: Computes Wilson score confidence interval for a proportion (k successes out of n trials)
- `snap_to_grid`: Snaps a continuous temperature value to the nearest value in the predefined temperature grid

In [ ]:
def wilson_ci(k: int, n: int, alpha: float = 0.05) -> tuple:
    """Wilson score interval for a proportion."""
    z = norm.ppf(1 - alpha / 2)
    p_hat = k / n
    denom = 1 + z**2 / n
    center = (p_hat + z**2 / (2 * n)) / denom
    half = z * math.sqrt(p_hat * (1 - p_hat) / n + z**2 / (4 * n**2)) / denom
    return max(0.0, center - half), min(1.0, center + half)


def snap_to_grid(val: float, grid: list) -> float:
    return min(grid, key=lambda t: abs(t - val))

## Filter Valid OC Instances

An OC (only-correctable) instance is one where:
- The model initially answered incorrectly (`predict_is_oc_error == "true"`)
- The Arrhenius fit succeeded (non-empty, non-zero `predict_arrhenius_R2`)

In the full dataset, this yields exactly 30 instances. In the demo (5-instance mini subset), we get 5.

In [ ]:
examples = data["datasets"][0]["examples"]

oc_valid = [
    e for e in examples
    if e.get("predict_is_oc_error") == "true"
    and e.get("predict_arrhenius_R2", "") not in ("", None, "nan")
    and e.get("predict_arrhenius_R2", "0") != "0"
]
logger.info(f"Valid OC instances: {len(oc_valid)} (demo expects {N_VALID_OC_EXPECTED})")
assert len(oc_valid) == N_VALID_OC_EXPECTED, f"Expected {N_VALID_OC_EXPECTED} valid OC instances, got {len(oc_valid)}"

## Metric 1: Delta-Approx BON-16 Accuracy

Instead of requiring the full 350-sample Arrhenius regression to determine the optimal temperature `T_operating`, this metric approximates it from a **single forward pass**:

```
T_thresh ≈ Delta / ln(16)  where Delta = logit(wrong) - logit(correct)
T_operating = max(T_thresh + 0.3, 0.05)  # clamp to grid minimum
```

The Best-of-16 accuracy is then `1 - (1 - p_correct(T_snap))^16`.

**Key finding**: Delta-approx BON-16 = 85.8% vs regression Ea BON-16 = 90.0% — a 4.2pp degradation attributable to the weak ρ(Ea,Δ)=0.106 correlation.

In [ ]:
def metric1_delta_approx(oc_valid: list) -> dict:
    """Compute T_operating(Delta-approx) Best-of-16 accuracy."""
    logger.info("Metric 1: Delta-approx BON-16 accuracy")
    bon_accs = []
    skipped = 0

    for inst in oc_valid:
        delta_str = inst.get("predict_Delta", "")
        if delta_str in ("", None, "nan"):
            logger.warning(f"Missing predict_Delta for qid={inst.get('metadata_question_id')}")
            skipped += 1
            continue

        delta = float(delta_str)
        t_thresh_approx = delta / math.log(N_BON)  # may be negative
        t_op_approx = t_thresh_approx + 0.3
        # clip to grid range [0.05, 1.0]
        t_op_approx = max(t_op_approx, GRID[0])
        t_snap = snap_to_grid(t_op_approx, GRID)

        p_correct_raw = inst.get("predict_p_correct_by_T", "{}")
        try:
            p_correct_dict = json.loads(p_correct_raw)
        except (json.JSONDecodeError, TypeError):
            logger.warning(f"Cannot parse p_correct_by_T for qid={inst.get('metadata_question_id')}")
            skipped += 1
            continue

        # Try multiple key formats
        p = None
        for key in [str(t_snap), f"{t_snap:.2f}", f"{t_snap:.1f}", f"{t_snap:.4f}"]:
            if key in p_correct_dict:
                p = float(p_correct_dict[key])
                break
        if p is None:
            # Try matching any key close to t_snap
            for k, v in p_correct_dict.items():
                try:
                    if abs(float(k) - t_snap) < 1e-9:
                        p = float(v)
                        break
                except ValueError:
                    continue
        if p is None:
            logger.warning(f"Key {t_snap} not found in p_correct_by_T keys={list(p_correct_dict.keys())} for qid={inst.get('metadata_question_id')}")
            p = 0.0

        bon = 1.0 - (1.0 - p) ** N_BON
        bon_accs.append(bon)
        logger.debug(f"qid={inst.get('metadata_question_id')} delta={delta:.4f} t_thresh={t_thresh_approx:.4f} t_op={t_op_approx:.4f} t_snap={t_snap} p={p:.4f} bon={bon:.4f}")

    accuracy = float(np.mean(bon_accs)) if bon_accs else 0.0
    logger.info(f"Delta-approx BON-16 accuracy: {accuracy:.4f} ({accuracy*100:.1f}%) over {len(bon_accs)} instances ({skipped} skipped)")
    return {
        "T_operating_delta_approx_BON16": accuracy,
        "n_instances": len(bon_accs),
        "n_skipped": skipped,
        "note": "Uses single-call Delta instead of regression Ea; T_thresh=Delta/ln(16)+0.3, snapped to grid {0.05,0.1,0.2,0.3,0.5,0.7,1.0}",
        "comparison_to_regression_Ea": "T_operating(regression Ea, delta=0.3) = 0.900",
    }


m1 = metric1_delta_approx(oc_valid)
print(f"\nMetric 1 result: {m1['T_operating_delta_approx_BON16']*100:.1f}% BON-16 accuracy (demo subset)")

## Metric 2: Wilson 95% Confidence Intervals on Table 1

The original paper reports point estimates for the fraction of instances where `T_operating` (simplified) acts as a lower bound on `T_pref` at each `N` (4, 8, 16, 32 samples).

Here we compute proper Wilson score 95% CIs. The Wilson interval is preferred over the naive Wald interval for small n and extreme proportions (k/n near 0 or 1).

**Key finding**: All lower CI bounds exceed the 60% criterion, even the wide N=4 CI [73.0%, 99.0%] (n=17).

In [ ]:
def metric2_wilson_cis(data: dict) -> dict:
    """Compute Wilson 95% CIs for Table 1 fractions."""
    logger.info("Metric 2: Wilson 95% CIs on Table 1 fractions")
    step6 = data["metadata"]["step6_T_thresh_validation"]["by_N"]
    results = {}

    for n_str, row in step6.items():
        n = int(row["n_total"])
        frac = float(row["fraction_simplified_is_lower_bound"])
        k = round(frac * n)
        # verify rounding
        if abs(k / n - frac) > 0.01:
            k_alt = k - 1 if k / n > frac else k + 1
            if abs(k_alt / n - frac) < abs(k / n - frac):
                k = k_alt
        lo, hi = wilson_ci(k, n)
        results[f"N{n_str}"] = {
            "k": k, "n": n, "fraction": frac,
            "wilson_lo": float(lo), "wilson_hi": float(hi),
            "display": f"{frac*100:.1f}% [{lo*100:.1f}%, {hi*100:.1f}%]",
        }
        logger.info(f"N={n_str}: k={k}/n={n} → {frac*100:.1f}% [{lo*100:.1f}%, {hi*100:.1f}%]")

    # Window fraction
    window_frac = float(data["metadata"]["step6_T_thresh_validation"]["window_fraction_T_TURN_above_T_thresh"])
    n_win = int(data["metadata"]["aggregate"]["n_instances"])
    k_win = round(window_frac * n_win)
    lo_w, hi_w = wilson_ci(k_win, n_win)
    results["window_fraction"] = {
        "k": k_win, "n": n_win, "fraction": window_frac,
        "wilson_lo": float(lo_w), "wilson_hi": float(hi_w),
        "display": f"{window_frac*100:.1f}% [{lo_w*100:.1f}%, {hi_w*100:.1f}%]",
    }
    logger.info(f"Window fraction: k={k_win}/n={n_win} → {window_frac*100:.1f}% [{lo_w*100:.1f}%, {hi_w*100:.1f}%]")
    return results


m2 = metric2_wilson_cis(data)
print("\nMetric 2 — Wilson CIs (full n=30 aggregate):")
for key, val in m2.items():
    print(f"  {key}: {val['display']}")

## Metric 3: Scatter Plot — T_operating vs T_pref

Two-panel figure:
- **Left panel**: Arrhenius `T_operating` (= `T_thresh_simplified + 0.3`) vs empirical `T_pref` (argmax accuracy across temperatures). Points colored by subject. Spearman ρ=0.674 (p=4.5e-5, n=30) confirms the Arrhenius model captures per-instance temperature preferences better than random.
- **Right panel**: Fixed `T=1.0` vs `T_pref` — all points pile on x=1.0, showing zero per-instance differentiation from a flat-temperature baseline.

In [ ]:
def metric3_scatter_plot(oc_valid: list) -> str:
    """Generate T_operating vs T_pref scatter plot."""
    logger.info("Metric 3: Scatter plot T_operating vs T_pref")

    subjects = sorted(set(e["metadata_subject"] for e in oc_valid))
    cmap = matplotlib.colormaps.get_cmap("tab20").resampled(len(subjects))
    subj_to_color = {s: cmap(i) for i, s in enumerate(subjects)}

    t_ops, t_prefs, colors = [], [], []
    for inst in oc_valid:
        t_thresh_simplified = float(inst["predict_T_thresh_N16_simplified"])
        t_op = t_thresh_simplified + 0.3
        t_pref = float(inst["predict_T_pref"])
        t_ops.append(t_op)
        t_prefs.append(t_pref)
        colors.append(subj_to_color[inst["metadata_subject"]])

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

    ax1.scatter(t_ops, t_prefs, c=colors, s=60, alpha=0.8, zorder=3)
    lo = min(min(t_ops) - 0.05, min(t_prefs) - 0.05)
    hi = max(max(t_ops) + 0.05, max(t_prefs) + 0.05)
    ax1.plot([lo, hi], [lo, hi], "k--", alpha=0.4, label="y=x (perfect)")
    ax1.set_xlabel("T_operating (Arrhenius, δ=0.3)", fontsize=11)
    ax1.set_ylabel("T_pref (empirical argmax)", fontsize=11)
    ax1.set_title("Arrhenius T_operating vs T_pref\nSpearman ρ=0.674 (p=4.5e-5), n=30", fontsize=11)
    for s in subjects:
        ax1.scatter([], [], c=[subj_to_color[s]], label=s, s=30)
    ax1.legend(fontsize=7, ncol=2, loc="upper left")

    ax2.scatter([1.0] * len(t_prefs), t_prefs, c=colors, s=60, alpha=0.8)
    ax2.axvline(1.0, color="gray", linestyle="--", alpha=0.5)
    ax2.set_xlabel("T_operating (fixed T=1.0)", fontsize=11)
    ax2.set_ylabel("T_pref (empirical argmax)", fontsize=11)
    ax2.set_title("Fixed T=1.0 vs T_pref\n(no per-instance differentiation)", fontsize=11)
    ax2.set_xlim([0.5, 1.5])

    plt.tight_layout()
    out_path = "t_operating_vs_t_pref.png"
    plt.savefig(out_path, dpi=150, bbox_inches="tight")
    plt.close()
    logger.info(f"Saved scatter plot to {out_path}")
    return out_path


scatter_path = metric3_scatter_plot(oc_valid)
print(f"\nMetric 3: Scatter plot saved to {scatter_path}")

## Metric 4: Power Analysis

Formal Fisher z-transform power analysis for the observed Spearman ρ=0.674 at n=30:
- Fisher z: `z = arctanh(ρ)`, SE = `1/sqrt(n-3)`
- Observed power = `Φ(|z|/SE - 1.96)`
- Minimum n for detecting ρ=0.4 at 80% power

**Key finding**: 98.9% observed power at n=30 — the study is well-powered for the confirmed result. Minimum n≈47 needed to detect a more modest ρ=0.4.

In [ ]:
def metric4_power_analysis() -> dict:
    """Formal power analysis via Fisher z-transform."""
    logger.info("Metric 4: Power analysis")
    rho_obs = 0.674
    n_obs = 30
    z_obs = float(np.arctanh(rho_obs))
    se = 1.0 / math.sqrt(n_obs - 3)
    z_alpha2 = 1.96
    z_beta_80 = 0.842

    # Observed power
    power_obs = float(norm.cdf(abs(z_obs) / se - z_alpha2))

    # Min n to detect rho=0.4 at 80% power
    rho_target = 0.4
    z_target = float(np.arctanh(rho_target))
    n_required = int(math.ceil(((z_alpha2 + z_beta_80) / z_target) ** 2 + 3))

    # Wilson CI for N=4 row (k=16, n=17)
    lo_n4, hi_n4 = wilson_ci(16, 17)

    logger.info(f"Observed power: {power_obs*100:.1f}% (n={n_obs}, ρ={rho_obs})")
    logger.info(f"n required for ρ=0.4 at 80% power: {n_required}")
    logger.info(f"N=4 Wilson CI: [{lo_n4*100:.1f}%, {hi_n4*100:.1f}%]")

    return {
        "observed_rho": rho_obs,
        "observed_n": n_obs,
        "fisher_z_obs": z_obs,
        "se": se,
        "observed_power_pct": float(power_obs * 100),
        "n_for_rho04_80pct_power": n_required,
        "n4_wilson_lo": float(lo_n4),
        "n4_wilson_hi": float(hi_n4),
        "interpretation": (
            f"The study is well-powered for the confirmed ρ={rho_obs} result "
            f"({power_obs*100:.1f}% power at n={n_obs}); "
            f"n≈{n_required} needed to detect ρ=0.4 at 80% power. "
            f"The N=4 Wilson CI [{lo_n4*100:.1f}%, {hi_n4*100:.1f}%] still exceeds the 60% criterion."
        ),
    }


m4 = metric4_power_analysis()
print(f"\nMetric 4: Observed power = {m4['observed_power_pct']:.1f}%")
print(f"  n required for ρ=0.4 at 80% power: {m4['n_for_rho04_80pct_power']}")
print(f"  Interpretation: {m4['interpretation']}")

## Metric 5: Deployment Algorithm

Three-case routing pseudocode for production deployment:
1. **Greedy-correct**: Model answers correctly at T=0 → return immediately (1 API call)
2. **OC instance**: Correct token visible in logprobs → compute `T_operating` from Delta, run N samples (1+N calls)
3. **Robust error**: Correct token not in top logprobs → fall back to fixed T=1.0 (1+N calls)

In [ ]:
DEPLOYMENT_PSEUDOCODE = '''\
def route_instance(prompt, N=16, logprob_threshold=-10.0):
    # CALL 1: greedy forward pass with logprobs (1 API call)
    greedy_out, logprobs = call_model(prompt, temperature=0, logprobs=True)
    correct_logprob = get_correct_token_logprob(logprobs)  # from answer options

    # Case 1: greedy already correct
    if greedy_out == correct_answer:
        return greedy_out  # 1 API call total

    # Case 2: OC instance — correct token visible in logprobs
    if correct_logprob > logprob_threshold:
        Delta = wrong_logprob - correct_logprob  # raw logit gap
        T_thresh = Delta / math.log(N)  # single-call approximation
        T_operating = max(T_thresh + 0.3, 0.3)  # clamp to grid minimum
        samples = call_model_n(prompt, T=T_operating, n=N)  # N API calls
        return majority_or_any_correct(samples)  # 1 + N API calls total

    # Case 3: robust error — correct token not in top logprobs
    # Fall back to fixed T=1.0 or TURN-adapted temperature
    samples = call_model_n(prompt, T=1.0, n=N)  # N API calls
    return majority_or_any_correct(samples)  # 1 + N API calls total

# Pilot gate cost amortization:
# Run calibration set of 50 instances to find logprob_threshold that
# maximizes recall of OC instances while excluding robust errors.
# This ~50-call cost is amortized over the full deployment set.
'''

print(DEPLOYMENT_PSEUDOCODE)

## Metric 6: Dataset Discrepancy Note

Clarifies the relationship between the catalysis set (50 items) and the main 500-item MMLU-Pro set.

In [ ]:
DATASET_DISCREPANCY_NOTE = (
    "The catalysis set (50 items) is a stratified subset of the 500-item main set; "
    "the temperature-sweep analysis (Sections 4.3–4.8) uses all 450 remaining "
    "main-set OC instances (not the 50 catalysis items) to avoid overlap between "
    "the catalysis conditioning and the primary Arrhenius analysis."
)

print("Dataset Discrepancy Note:")
print(DATASET_DISCREPANCY_NOTE)

## Per-Instance BON Scorer

Helper function to compute the delta-approx BON-16 score for a single instance — used when building the output dataset.

In [ ]:
def _compute_bon_for_instance(inst: dict) -> float:
    delta_str = inst.get("predict_Delta", "")
    if delta_str in ("", None, "nan"):
        return 0.0
    delta = float(delta_str)
    t_thresh_approx = delta / math.log(N_BON)
    t_op_approx = max(t_thresh_approx + 0.3, GRID[0])
    t_snap = snap_to_grid(t_op_approx, GRID)
    try:
        p_correct_dict = json.loads(inst.get("predict_p_correct_by_T", "{}"))
    except (json.JSONDecodeError, TypeError):
        return 0.0
    p = None
    for key in [str(t_snap), f"{t_snap:.2f}", f"{t_snap:.1f}"]:
        if key in p_correct_dict:
            p = float(p_correct_dict[key])
            break
    if p is None:
        p = 0.0
    return float(1.0 - (1.0 - p) ** N_BON)

## Assemble Output

Combines all metric results into the `eval_out` dictionary matching the experiment schema, then saves to `eval_out.json`.

In [ ]:
eval_out = {
    "metadata": {
        "evaluation_name": "Phi-4 Arrhenius Re-Analysis",
        "description": (
            "Zero-API-cost re-analysis of phi-4 Arrhenius experiment. "
            "Produces Delta-approx accuracy, Wilson CIs, scatter plot, "
            "power analysis, deployment pseudocode, and dataset discrepancy note."
        ),
        "source_experiment": "art_Ux9ENkZVYpvn",
        "n_valid_oc_instances": len(oc_valid),
        "delta_approx_accuracy": m1,
        "table1_wilson_cis": m2,
        "scatter_plot_path": scatter_path,
        "power_analysis": m4,
        "deployment_algorithm_pseudocode": DEPLOYMENT_PSEUDOCODE,
        "dataset_discrepancy_note": DATASET_DISCREPANCY_NOTE,
    },
    "metrics_agg": {
        "delta_approx_BON16_accuracy": m1["T_operating_delta_approx_BON16"],
        "delta_approx_BON16_accuracy_pct": round(m1["T_operating_delta_approx_BON16"] * 100, 1),
        "regression_Ea_BON16_accuracy_pct": 90.0,
        "accuracy_delta_vs_regression_pp": round(
            (m1["T_operating_delta_approx_BON16"] - 0.9) * 100, 1
        ),
        "N4_wilson_lo_pct": round(m2["N4"]["wilson_lo"] * 100, 1),
        "N4_wilson_hi_pct": round(m2["N4"]["wilson_hi"] * 100, 1),
        "N16_wilson_lo_pct": round(m2["N16"]["wilson_lo"] * 100, 1),
        "N32_wilson_lo_pct": round(m2["N32"]["wilson_lo"] * 100, 1),
        "window_fraction_wilson_lo_pct": round(m2["window_fraction"]["wilson_lo"] * 100, 1),
        "observed_power_pct": round(m4["observed_power_pct"], 1),
        "n_for_rho04_80pct_power": float(m4["n_for_rho04_80pct_power"]),
        "fisher_z_obs": round(m4["fisher_z_obs"], 4),
        "observed_rho": m4["observed_rho"],
    },
    "datasets": [
        {
            "dataset": "TIGER-Lab/MMLU-Pro",
            "examples": [
                {
                    "input": inst["input"],
                    "output": inst["output"],
                    "metadata_question_id": inst.get("metadata_question_id"),
                    "metadata_subject": inst.get("metadata_subject", ""),
                    "metadata_split": inst.get("metadata_split", ""),
                    "predict_is_oc_error": inst.get("predict_is_oc_error", ""),
                    "predict_arrhenius_Ea": inst.get("predict_arrhenius_Ea", ""),
                    "predict_arrhenius_R2": inst.get("predict_arrhenius_R2", ""),
                    "predict_Delta": inst.get("predict_Delta", ""),
                    "predict_T_pref": inst.get("predict_T_pref", ""),
                    "predict_T_thresh_N16_simplified": inst.get("predict_T_thresh_N16_simplified", ""),
                    "eval_bon16_delta_approx": _compute_bon_for_instance(inst),
                }
                for inst in oc_valid
            ],
        }
    ],
}

with open("eval_out.json", "w") as f:
    json.dump(eval_out, f, indent=2)

import os
print(f"Saved eval_out.json ({os.path.getsize('eval_out.json') / 1024:.1f} KB)")

## Results Summary

Visual summary of all key metrics from the re-analysis.

In [ ]:
matplotlib.use("Agg")  # ensure non-interactive backend

print("=" * 60)
print("PHI-4 ARRHENIUS RE-ANALYSIS — RESULTS SUMMARY")
print("=" * 60)

print("\n[Metric 1] Delta-Approx BON-16 Accuracy")
print(f"  Delta-approx:    {m1['T_operating_delta_approx_BON16']*100:.1f}%  (demo subset: n={m1['n_instances']})")
print(f"  Regression Ea:   90.0%  (full n=30, from original experiment)")
print(f"  Gap:             {(m1['T_operating_delta_approx_BON16']-0.9)*100:.1f}pp  (due to weak ρ(Ea,Δ)=0.106)")

print("\n[Metric 2] Wilson 95% CIs on Table 1 (full n=30 aggregate)")
rows = [
    ("N=4",  m2["N4"]),
    ("N=8",  m2["N8"]),
    ("N=16", m2["N16"]),
    ("N=32", m2["N32"]),
    ("Window", m2["window_fraction"]),
]
print(f"  {'Condition':<10} {'k/n':<8} {'Fraction':>9} {'Wilson 95% CI':>20}")
print(f"  {'-'*10} {'-'*8} {'-'*9} {'-'*20}")
for label, row in rows:
    ci = f"[{row['wilson_lo']*100:.1f}%, {row['wilson_hi']*100:.1f}%]"
    print(f"  {label:<10} {row['k']}/{row['n']:<6} {row['fraction']*100:>8.1f}%  {ci:>20}")

print("\n[Metric 3] Scatter Plot")
print(f"  Saved to: {scatter_path}")
print(f"  Left: Arrhenius T_op vs T_pref (Spearman ρ=0.674, p=4.5e-5, n=30)")
print(f"  Right: Fixed T=1.0 vs T_pref (zero per-instance differentiation)")

print("\n[Metric 4] Power Analysis")
print(f"  Observed power:  {m4['observed_power_pct']:.1f}%  (ρ=0.674, n=30)")
print(f"  Fisher z:        {m4['fisher_z_obs']:.4f}")
print(f"  n for ρ=0.4 @ 80% power: {m4['n_for_rho04_80pct_power']}")

print("\n[Verdict]")
print("  C3 CONFIRMED: T_simplified is a lower bound for T_pref at all N")
print("  C4 CONFIRMED: Window fraction 93.3% [78.7%, 98.2%] >> 70%")
print("  C2 NOT MET:   ρ(Ea,Δ)=0.106 — Delta is a poor proxy for Ea")
print("  => 4.2pp accuracy gap when using single-call Delta vs regression Ea")

# Bar chart of key accuracy metrics
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Left: accuracy comparison
ax = axes[0]
labels = ["Regression Ea\n(full experiment)", "Delta-Approx\n(single call)"]
values = [90.0, m1["T_operating_delta_approx_BON16"] * 100]
colors_bar = ["#2196F3", "#FF9800"]
bars = ax.bar(labels, values, color=colors_bar, alpha=0.8, edgecolor="black", linewidth=0.8)
ax.axhline(60, color="red", linestyle="--", alpha=0.6, label="60% criterion")
ax.set_ylabel("BON-16 Accuracy (%)", fontsize=11)
ax.set_title("BON-16 Accuracy: Regression Ea vs Delta-Approx", fontsize=11)
ax.set_ylim([0, 100])
ax.legend(fontsize=9)
for bar, v in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width()/2, v + 1, f"{v:.1f}%", ha="center", va="bottom", fontsize=10)

# Right: Wilson CIs
ax2 = axes[1]
ci_labels = ["N=4", "N=8", "N=16", "N=32", "Window"]
ci_keys = ["N4", "N8", "N16", "N32", "window_fraction"]
fractions = [m2[k]["fraction"] * 100 for k in ci_keys]
lo_bounds = [m2[k]["wilson_lo"] * 100 for k in ci_keys]
hi_bounds = [m2[k]["wilson_hi"] * 100 for k in ci_keys]
x = range(len(ci_labels))
ax2.bar(x, fractions, color="#4CAF50", alpha=0.7, edgecolor="black", linewidth=0.8, label="Point estimate")
ax2.errorbar(
    x, fractions,
    yerr=[np.array(fractions) - np.array(lo_bounds), np.array(hi_bounds) - np.array(fractions)],
    fmt="none", color="black", capsize=5, linewidth=1.5, label="Wilson 95% CI"
)
ax2.axhline(60, color="red", linestyle="--", alpha=0.6, label="60% criterion")
ax2.set_xticks(list(x))
ax2.set_xticklabels(ci_labels)
ax2.set_ylabel("Fraction T_simplified is lower bound (%)", fontsize=10)
ax2.set_title("Wilson 95% CIs — Table 1 (n=30 aggregate)", fontsize=11)
ax2.set_ylim([0, 110])
ax2.legend(fontsize=9)

plt.tight_layout()
plt.savefig("results_summary.png", dpi=120, bbox_inches="tight")
plt.close()
print("\nResults summary chart saved to results_summary.png")